# Step 6 — 10 videos + labels → your own batch_001.npz (Phase A complete!)

**Goal:** encode several videos, attach labels, stack, and save a `.npz` in the author's exact format:
- `data`   : `(num_videos, 100, 20, 4096)`  float16
- `labels` : `(num_videos, 2)`               float16  (`[1,0]`=no-accident, `[0,1]`=accident)

⚠️ Encoding is slow on CPU. Start small with the knobs below, confirm it works, then scale up.
- `N_ACC`, `N_NON` = how many accident / non-accident videos
- `MAX_FRAMES`     = frames per video (100 = full)

Output goes to `Dataset/features_generated/testing/` — NOT over the author's files.

## 1. Models + helpers + the per-video function

In [ ]:
import cv2, time, glob, os
import numpy as np
import torch
import torch.nn as nn

N_FRAMES=100; N_DETECTION=20; N_OBJECTS=19; FEAT_DIM=4096
KEEP={1,2,3,4,6,7,8}; SCORE_THRESH=0.5
IMAGENET_MEAN=torch.tensor([0.485,0.456,0.406]).view(1,3,1,1)
IMAGENET_STD =torch.tensor([0.229,0.224,0.225]).view(1,3,1,1)

from torchvision.models.detection import fasterrcnn_resnet50_fpn
try:
    from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights
    detector = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT)
except Exception:
    detector = fasterrcnn_resnet50_fpn(pretrained=True)
detector.eval()

from torchvision.models import vgg16
try:
    from torchvision.models import VGG16_Weights
    vgg = vgg16(weights=VGG16_Weights.IMAGENET1K_V1)
except Exception:
    vgg = vgg16(pretrained=True)
vgg.eval()
vgg_features=vgg.features; vgg_avgpool=vgg.avgpool
vgg_fc7=nn.Sequential(*list(vgg.classifier.children())[:5])

def detect_boxes(frame):
    t=torch.from_numpy(frame).permute(2,0,1).float()/255.0
    with torch.no_grad():
        out=detector([t])[0]
    kept=[(float(s),b) for b,l,s in zip(out['boxes'].numpy(),out['labels'].numpy(),out['scores'].numpy())
          if s>=SCORE_THRESH and int(l) in KEEP]
    kept.sort(key=lambda z:z[0], reverse=True)
    return [b for _,b in kept[:N_OBJECTS]]

def crop_box(frame, box):
    h,w,_=frame.shape; x1,y1,x2,y2=box
    x1=max(0,int(round(x1))); y1=max(0,int(round(y1)))
    x2=min(w,int(round(x2))); y2=min(h,int(round(y2)))
    return None if (x2-x1<2 or y2-y1<2) else frame[y1:y2,x1:x2]

def to_features_batch(crops):
    if not crops: return np.zeros((0,FEAT_DIM),np.float32)
    batch=np.stack([cv2.resize(c,(224,224)) for c in crops]).astype(np.float32)/255.0
    x=torch.from_numpy(batch).permute(0,3,1,2)
    x=(x-IMAGENET_MEAN)/IMAGENET_STD
    with torch.no_grad():
        x=vgg_features(x); x=vgg_avgpool(x); x=torch.flatten(x,1); x=vgg_fc7(x)
    return x.numpy().astype(np.float32)

def encode_video(path, max_frames=100):
    cap=cv2.VideoCapture(path); frames=[]
    while True:
        ok,f=cap.read()
        if not ok: break
        frames.append(cv2.cvtColor(f,cv2.COLOR_BGR2RGB))
    cap.release()
    out=np.zeros((N_FRAMES,N_DETECTION,FEAT_DIM),np.float32)
    for t in range(min(max_frames,len(frames))):
        frame=frames[t]; crops=[frame]
        for box in detect_boxes(frame):
            c=crop_box(frame,box)
            if c is not None: crops.append(c)
            if len(crops)==N_DETECTION: break
        feats=to_features_batch(crops)
        out[t,:feats.shape[0],:]=feats
    return out
print('ready.')

## 2. Pick the videos + their labels
Label rule: Accident -> [0,1], NonAccident -> [1,0].

In [ ]:
N_ACC = 1          # accident videos   (start small!)
N_NON = 1          # non-accident videos
MAX_FRAMES = 20    # frames per video  (100 = full)

acc = sorted(glob.glob('../Dataset/videos/test/Accident/*.mp4'))[:N_ACC]
non = sorted(glob.glob('../Dataset/videos/test/NonAccident/*.mp4'))[:N_NON]

items = [(p,(0.0,1.0)) for p in acc] + [(p,(1.0,0.0)) for p in non]
print(f'{len(items)} videos:')
for p,lab in items:
    print('  ', os.path.basename(p), '->', 'ACCIDENT' if lab[1]==1 else 'no-accident')

## 3. Encode all videos into data + labels arrays

In [ ]:
num=len(items)
data=np.zeros((num,N_FRAMES,N_DETECTION,FEAT_DIM),np.float32)
labels=np.zeros((num,2),np.float32)

t0=time.time()
for i,(path,lab) in enumerate(items):
    print(f'[{i+1}/{num}] {os.path.basename(path)} ...', end=' ', flush=True)
    ti=time.time()
    data[i]=encode_video(path, MAX_FRAMES)
    labels[i]=lab
    print(f'done in {time.time()-ti:.1f}s')
print(f'\nTOTAL {time.time()-t0:.1f}s   data={data.shape} labels={labels.shape}')

## 4. Save as float16 .npz (author's format)

In [ ]:
out_dir='../Dataset/features_generated/testing'
os.makedirs(out_dir, exist_ok=True)
out_path=os.path.join(out_dir,'batch_001.npz')

np.savez(out_path,
         data=data.astype(np.float16),      # author stores float16
         labels=labels.astype(np.float16))
print('saved:', out_path)
print('size :', round(os.path.getsize(out_path)/1e6,2), 'MB')

## 5. Reload and sanity-check

In [ ]:
f=np.load(out_path)
print('keys  :', f.files)
print('data  :', f['data'].shape, f['data'].dtype)
print('labels:', f['labels'].shape, f['labels'].dtype)
print('labels values:\n', f['labels'])
print('data >= 0 ?', bool((f['data']>=0).all()), ' max:', round(float(f['data'].max()),2))

## What to take away — PHASE A DONE
You just produced a `.npz` in the author's exact schema from raw video. That is the missing feature-creation step, end to end.

**To scale up:** set `N_ACC=5, N_NON=5` (=10 videos) and `MAX_FRAMES=100`, re-run cells 2–5. (Expect it to take a while on CPU.)

Next: **Step 7 — compare your `.npz` against the author's reference** to see how faithful it is.